# PPRL TOOLKIT NOTEBOOK TUTORIAL 2026

## How to install PPRL Toolkit

In [1]:
%pip install "numpy<2"
%pip install -e .

Note: you may need to restart the kernel to use updated packages.
Obtaining file:///home/lefteris/pprl_toolkit
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for pprl (pyproject.toml) ... done
  Created wheel for pprl: filename=pprl-0.1.0-0.editable-py3-none-any.whl size=6822 sha256=0a5f4591b55af5ea128f03b31bd2830f61b62e46dadce3d143b022f624ef1e9e
  Stored in directory: /tmp/pip-ephem-wheel-cache-tctg4bfi/wheels/3f/13/a5/bb14a4ee961501247d1c335a95e7ef9692213c776834c0570b
Successfully built pprl
  Attempting uninstall: pprl
    Found existing installation: pprl 0.1.0
    Uninstalling pprl-0.1.0:
      Successfully uninstalled pprl-0.1.0
Note: you may need to restart the kernel to use updated packages.


## Load Dataset

In [3]:
import pandas as pd

abt = pd.read_csv("data/D2/abtclean.csv", sep='|')
buy = pd.read_csv("data/D2/buyclean.csv", sep='|')

abt.head(10)

,id,name,description,price
0,0,Sony Turntable - PSLX350H,Sony Turntable - PSLX350H/ Belt Drive System/ ...,NaN
1,1,Bose Acoustimass 5 Series III Speaker System -...,Bose Acoustimass 5 Series III Speaker System -...,399.0
2,2,Sony Switcher - SBV40S,Sony Switcher - SBV40S/ Eliminates Disconnecti...,49.0
3,3,Sony 5 Disc CD Player - CDPCE375,Sony 5 Disc CD Player- CDPCE375/ 5 Disc Change...,NaN
4,4,Bose 27028 161 Bookshelf Pair Speakers In Whit...,Bose 161 Bookshelf Speakers In White - 161WH/ ...,158.0
5,5,Denon Stereo Tuner - TU1500RD,Denon Stereo Tuner - TU1500RD/ RDS Radio Data ...,375.0
6,6,KitchenAid Pasta Roller And Cutter - KPRA,KitchenAid Pasta Roller And Cutter - KPRA/ One...,NaN
7,7,Panasonic Yeast Pro Automatic Breadmaker - SDY...,Panasonic Yeast Pro Automatic Breadmaker - SDY...,NaN
8,8,Sony Vertical-In-The-Ear Stereo Headphones - M...,Sony Vertical-In-The-Ear Stereo Headphones - M...,NaN
9,9,Panasonic 2-Line Integrated Telephone - KXTSC14W,Panasonic 2-Line Integrated Telephone - KXTSC1...,NaN


In [4]:
buy.head(10)

,id,name,description,price
0,0,Linksys EtherFast EZXS88W Ethernet Switch - EZ...,Linksys EtherFast 8-Port 10/100 Switch (New/Wo...,NaN
1,1,Linksys EtherFast EZXS55W Ethernet Switch,5 x 10/100Base-TX LAN,NaN
2,2,Netgear ProSafe FS105 Ethernet Switch - FS105NA,NETGEAR FS105 Prosafe 5 Port 10/100 Desktop Sw...,NaN
3,3,Belkin Pro Series High Integrity VGA/SVGA Moni...,1 x HD-15 - 1 x HD-15 - 10ft - Beige,NaN
4,4,Netgear ProSafe JFS516 Ethernet Switch,Netgear ProSafe 16 Port 10/100 Rackmount Switc...,NaN
5,5,LaCie Pocket Floppy Disk Drive - 706018,LaCie Pocket USB Floppy 1.44 MB,NaN
6,6,Canon KP 36IP Print Cartridge / Paper Kit - 77...,36 Page 4' x 6',9.99
7,7,Kensington Orbit Optical Trackball - USB w/PS2...,Optical - USB PS/2,NaN
8,8,Linksys EtherFast EF4116 Ethernet Switch,16 x 10/100Base-TX LAN,NaN
9,9,Linksys EtherFast EF4124 Ethernet Switch,24 x 10/100Base-TX LAN,64.99


## Creating and assigning a feature factory

The next step is to decide how to process each of the columns in our datasets. The pprl.embedder.features module provides functions that process different data types so that they can be embedded into the Bloom filter. We pass these functions into the embedder in a dictionary called a feature factory. We also provide a column specification for each dataset mapping our columns to column types in the factory.

In [5]:
from pprl.embedder import features
from functools import partial


# Map the columns on which we want to generate features to the corresponding feature generation functions
factory = {'name' : features.gen_name_features}
spec_abt = {'name' : 'name'}
spec_buy = {'name' : 'name'}

## Embedding the data

With our specifications sorted out, we can get to creating our Bloom filter embedding. We can create our Embedder instance and use it to embed our data with their column specifications. The Embedder object has two more parameters: the size of the filter and the number of hashes. We can use the defaults.

In [6]:
from pprl.embedder import Embedder

embedder : Embedder = Embedder(factory, bf_size=1024, num_hashes=12)
abt_embedded = embedder.embed(abt, colspec=spec_abt, update_thresholds=True)
buy_embedded = embedder.embed(buy, colspec=spec_buy, update_thresholds=True)

## Performing the linkage

We can now perform the linkage by comparing these Bloom filter embeddings. The package uses the Soft Cosine Measure to calculate record-wise similarity scores.

In [9]:
similarities = embedder.compare(abt_embedded, buy_embedded)

similarities[:10]

SimilarityArray([[0.50735052, 0.50183481, 0.52572983, ..., 0.46103728,
                  0.4766221 , 0.51545246],
                 [0.59968444, 0.59800037, 0.60950366, ..., 0.56317602,
                  0.62043966, 0.66574252],
                 [0.58039038, 0.58353049, 0.57355936, ..., 0.40130631,
                  0.43276806, 0.48136776],
                 ...,
                 [0.593624  , 0.59174905, 0.60563242, ..., 0.56330144,
                  0.59629681, 0.65330582],
                 [0.60723842, 0.6105238 , 0.6400589 , ..., 0.56872047,
                  0.58676706, 0.66284995],
                 [0.61047363, 0.60871791, 0.58898   , ..., 0.56197696,
                  0.60479893, 0.66890357]])

Lastly, we compute the matching using an adapted Hungarian algorithm with local match thresholds:

In [24]:
matching = similarities.match()

print(f"Returns {len(matching)} arrays one for ABT and one for BUY")
print(f"Where each item of ABT is matched with each item of BUY.\nPrinting the shapes of the 2 arrays: {matching[0].shape}, {matching[1].shape}")

Returns 2 arrays one for ABT and one for BUY
Where each item of ABT is matched with each item of BUY.
Printing the shapes of the 2 arrays: (321,), (321,)


## Evaluation

In [32]:
ground_truth = pd.read_csv("data/D2/gtclean.csv", sep='|')

true_positives = 0
total_matching_pairs = matching[0].shape[0]

for _, (id1, id2) in ground_truth.iterrows():
    if id1 in matching[0]:
        id1_indices = (matching[0] == id1).nonzero()
        for id1_index in id1_indices:
            if id2 == matching[1][id1_index]:
                true_positives += 1

num_of_true_duplicates = ground_truth.shape[0]
false_negatives = num_of_true_duplicates - true_positives
false_positives = total_matching_pairs - true_positives
cardinality = abt.shape[0] * buy.shape[0]
true_negatives = cardinality - false_negatives - num_of_true_duplicates
precision = true_positives / total_matching_pairs
recall = true_positives / num_of_true_duplicates
if precision == 0.0 or recall == 0.0:
    f1 = 0.0
else:
    f1 = 2*((precision*recall)/(precision+recall))

print(f"Precision: {precision}\nRecall: {recall}\nF1: {f1}")

Precision: 0.5482866043613707
Recall: 0.16541353383458646
F1: 0.25415162454873647
